# Inside Airbnb: Munich listing points

This Colab-style notebook reads Inside Airbnb listing point data for Munich, converts the latitude/longitude columns into a GeoDataFrame, and visualizes the points on a Folium map.

**Learning goals**

- Download open tabular geodata from Inside Airbnb.
- Convert CSV latitude/longitude fields into point geometries.
- Inspect basic listing attributes such as room type, price, and neighbourhood.
- Map listing points together with neighbourhood boundaries.
- Save a clean GeoJSON for later joins with OSM, Zensus, Mapillary, or other course datasets.

**Data source**

- Inside Airbnb Get the Data page: https://insideairbnb.com/get-the-data/
- Data license on the page: Creative Commons Attribution 4.0 International License.

Inside Airbnb data is useful for teaching and exploratory analysis, but it is scraped open data. Treat it as a research dataset with known data-quality limitations, not an official registry.

In [ ]:
# Colab setup
!pip -q install geopandas shapely pyproj folium mapclassify beautifulsoup4 requests

In [ ]:
from pathlib import Path
from urllib.parse import urljoin

import folium
import geopandas as gpd
import pandas as pd
import requests
from bs4 import BeautifulSoup
from folium.plugins import MarkerCluster
from shapely.geometry import Point

## 1. Configuration

The notebook uses the lightweight `visualisations/listings.csv` file because it already contains latitude and longitude and is small enough for classroom mapping. It also downloads `neighbourhoods.geojson` for boundaries.

In [ ]:
INSIDE_AIRBNB_DATA_PAGE = "https://insideairbnb.com/get-the-data/"
CITY_LABEL = "Munich"
CACHE_DIR = Path("/content/inside_airbnb_munich")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Fallback URLs from the current Inside Airbnb Munich release shown on the Get the Data page.
# The link finder below should normally pick the current URLs automatically.
FALLBACK_URLS = {
    "summary_listings": "https://data.insideairbnb.com/germany/bv/munich/2025-09-27/visualisations/listings.csv",
    "neighbourhoods_geojson": "https://data.insideairbnb.com/germany/bv/munich/2025-09-27/visualisations/neighbourhoods.geojson",
}

## 2. Find current Munich download links

Inside Airbnb updates city snapshots periodically. This helper reads the official data page and finds the current Munich links. If the page layout changes, the notebook falls back to the known Munich release URLs above.

In [ ]:
def find_inside_airbnb_city_links(city_label=CITY_LABEL, page_url=INSIDE_AIRBNB_DATA_PAGE):
    response = requests.get(page_url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    city_heading = None
    for heading in soup.find_all(["h2", "h3", "h4"]):
        if city_label.lower() in heading.get_text(" ", strip=True).lower():
            city_heading = heading
            break

    if city_heading is None:
        raise ValueError(f"Could not find city heading for {city_label!r}")

    links = {}
    for node in city_heading.find_all_next():
        if node.name in ["h2", "h3", "h4"] and node is not city_heading:
            break
        if node.name != "a":
            continue
        href = node.get("href") or ""
        text = node.get_text(" ", strip=True).lower()
        url = urljoin(page_url, href)
        url_lower = url.lower()

        if text == "listings.csv" and "/visualisations/" in url_lower:
            links["summary_listings"] = url
        elif text == "neighbourhoods.geojson" and "/visualisations/" in url_lower:
            links["neighbourhoods_geojson"] = url
        elif text == "listings.csv.gz" and "/data/" in url_lower:
            links["detailed_listings"] = url

    missing = {"summary_listings", "neighbourhoods_geojson"} - set(links)
    if missing:
        raise ValueError(f"Missing expected links for {city_label}: {sorted(missing)}")
    return links


try:
    airbnb_links = find_inside_airbnb_city_links()
except Exception as exc:
    print(f"Using fallback URLs because link discovery failed: {exc}")
    airbnb_links = FALLBACK_URLS.copy()

airbnb_links

## 3. Read listing points

The summary listings table contains one row per listing and already includes `latitude` and `longitude`. The loader keeps the raw table, standardizes a numeric `price_eur` column when possible, and returns a GeoDataFrame in WGS84 (`EPSG:4326`).

In [ ]:
def read_munich_listings(url=airbnb_links["summary_listings"]):
    df = pd.read_csv(url)
    required = {"id", "latitude", "longitude"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Listings table is missing required columns: {sorted(missing)}")

    df = df.dropna(subset=["latitude", "longitude"]).copy()
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["latitude", "longitude"])

    if "price" in df.columns:
        df["price_eur"] = (
            df["price"].astype(str)
            .str.replace("€", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df["price_eur"] = pd.to_numeric(df["price_eur"], errors="coerce")

    geometry = gpd.points_from_xy(df["longitude"], df["latitude"])
    return gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")


gdf_listings = read_munich_listings()
print(f"Munich listings with coordinates: {len(gdf_listings):,}")
gdf_listings.head()

## 4. Quick data checks

In [ ]:
print("Columns:")
print(list(gdf_listings.columns))

if "room_type" in gdf_listings.columns:
    display(gdf_listings["room_type"].value_counts(dropna=False).to_frame("count"))

if "price_eur" in gdf_listings.columns:
    display(gdf_listings["price_eur"].describe().to_frame("price_eur"))

if "neighbourhood" in gdf_listings.columns:
    display(gdf_listings["neighbourhood"].value_counts().head(10).to_frame("listings"))

## 5. Read neighbourhood boundaries

In [ ]:
def read_munich_neighbourhoods(url=airbnb_links["neighbourhoods_geojson"]):
    neighbourhoods = gpd.read_file(url).to_crs("EPSG:4326")
    return neighbourhoods


gdf_neighbourhoods = read_munich_neighbourhoods()
print(f"Munich neighbourhood polygons: {len(gdf_neighbourhoods):,}")
gdf_neighbourhoods.head()

## 6. Map listing points

For browser performance, the map samples points when the dataset is large. Change `MAX_POINTS_ON_MAP` if you want to display more.

In [ ]:
ROOM_TYPE_COLORS = {
    "Entire home/apt": "#1f77b4",
    "Private room": "#2ca02c",
    "Shared room": "#ff7f0e",
    "Hotel room": "#9467bd",
}


def map_airbnb_points(gdf, neighbourhoods=None, max_points=1500):
    center = gdf.geometry.unary_union.centroid
    m = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

    if neighbourhoods is not None and len(neighbourhoods):
        folium.GeoJson(
            neighbourhoods[["geometry"]],
            name="Inside Airbnb neighbourhoods",
            style_function=lambda feature: {
                "color": "#444444",
                "weight": 1,
                "fillOpacity": 0.02,
            },
        ).add_to(m)

    plot_gdf = gdf.copy()
    if len(plot_gdf) > max_points:
        plot_gdf = plot_gdf.sample(max_points, random_state=42)

    cluster = MarkerCluster(name=f"Listings sample ({len(plot_gdf):,})").add_to(m)
    for _, row in plot_gdf.iterrows():
        room_type = row.get("room_type", "unknown")
        color = ROOM_TYPE_COLORS.get(room_type, "#d62728")
        price = row.get("price_eur", row.get("price", ""))
        popup = f"<b>{row.get('name', 'listing')}</b><br>room_type: {room_type}<br>price: {price}<br>neighbourhood: {row.get('neighbourhood', '')}"
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.75,
            popup=folium.Popup(popup, max_width=260),
        ).add_to(cluster)

    folium.LayerControl().add_to(m)
    return m


MAX_POINTS_ON_MAP = 1500
map_airbnb_points(gdf_listings, gdf_neighbourhoods, max_points=MAX_POINTS_ON_MAP)

## 7. Optional: spatial join to neighbourhood polygons

The CSV already includes a neighbourhood field. This spatial join demonstrates how to assign polygon attributes from geometry, which is the same pattern used for joins with Zensus grids or administrative boundaries.

In [ ]:
if len(gdf_neighbourhoods):
    polygon_cols = [col for col in gdf_neighbourhoods.columns if col != "geometry"]
    keep_cols = polygon_cols[:2] + ["geometry"]
    joined = gpd.sjoin(
        gdf_listings,
        gdf_neighbourhoods[keep_cols],
        how="left",
        predicate="within",
    )
    print(f"Spatially joined rows: {len(joined):,}")
    display(joined.head())
else:
    joined = gdf_listings.copy()
    print("No neighbourhood polygons available for spatial join.")

## 8. Save prepared point data

In [ ]:
out_geojson = CACHE_DIR / "airbnb_munich_listings.geojson"
gdf_listings.to_file(out_geojson, driver="GeoJSON")
out_geojson